# 05 — Compute Player Skill Features
Compute leakage-safe cumulative player shooting history before each shot across all prior
events, then save enriched train/test datasets for later modeling.

Player history is accumulated from all prior shots across all five leagues, using
chronological order only — it is never derived from split membership.

**New features added:**
- `career_shots_before_shot` — prior shot attempts by this player
- `career_goals_before_shot` — prior goals scored by this player
- `career_non_goal_shots_before_shot` — prior non-goal shots by this player
- `career_conversion_rate_before_shot` — prior goal rate; 0.0 when no history
- `has_prior_shot_history` — 1 if any prior history exists, else 0

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display

## Step 0: Path validation

In [ ]:
DATA_DIR   = Path("../data")
TRAIN_PATH = DATA_DIR / "wyscout_train_xg_baseline_predictions.parquet"
TEST_PATH  = DATA_DIR / "wyscout_test_xg_baseline_predictions.parquet"

assert TRAIN_PATH.exists(), "Run 04_train_baseline_model.ipynb first"
assert TEST_PATH.exists(),  "Run 04_train_baseline_model.ipynb first"
print("Paths OK")

## Step 1: Load baseline prediction tables

In [ ]:
train = pd.read_parquet(TRAIN_PATH)
test  = pd.read_parquet(TEST_PATH)

REQUIRED_COLS = {
    "league", "matchId", "playerId", "teamId",
    "eventSec", "matchPeriod",
    "match_datetime_utc", "period_sort_key",
    "seconds_from_match_start", "shot_sequence_in_match",
    "is_goal", "xg_pred",
    "is_penalty", "is_direct_free_kick",  # confirms notebooks 01-04 ran
}

assert REQUIRED_COLS.issubset(train.columns), \
    f"Missing train columns: {REQUIRED_COLS - set(train.columns)}"
assert REQUIRED_COLS.issubset(test.columns), \
    f"Missing test columns: {REQUIRED_COLS - set(test.columns)}"
assert set(train["league"].unique()) == {"France", "Germany", "Italy", "Spain"}, \
    "Wrong leagues in train"
assert set(test["league"].unique()) == {"England"}, "Wrong leagues in test"
assert len(train) > 0 and len(test) > 0, "Empty train or test set"

# Diagnostic: player-overlap check (expected zero for domestic-league dataset)
# If overlap > 0, those players' histories will correctly span both splits — not an error.
train_players  = set(train["playerId"].dropna().unique())
test_players   = set(test["playerId"].dropna().unique())
player_overlap = train_players & test_players
print(f"Players in both splits: {len(player_overlap)}  (expected 0 for domestic-league data)")

# Pipeline continuity note: xg_pred is preserved from notebook 04 and is NOT used
# in cumulative skill computations — it is carried forward for downstream use.

summary = pd.DataFrame({
    "split":     ["train", "test"],
    "rows":      [len(train), len(test)],
    "goal_rate": [train["is_goal"].mean(), test["is_goal"].mean()],
})
display(summary)

# Add split marker before concat — used to split back after feature computation
train = train.copy()
test  = test.copy()
train["dataset_split"] = "train"
test["dataset_split"]  = "test"

## Step 2: Build a combined chronological table

Cumulative features must be computed on the combined dataset sorted globally.
Splitting and computing separately would miss valid prior history for England players
(not an issue here given league-split design, but required for correctness in principle).

In [ ]:
SORT_COLS = [
    "match_datetime_utc",
    "matchId",
    "period_sort_key",
    "seconds_from_match_start",
    "shot_sequence_in_match",
    "league",
    "playerId",
    "teamId",
]

shots = pd.concat([train, test], ignore_index=True)

# Null checks before sort/groupby — null values cause silent misbehaviour
for col in SORT_COLS + ["is_goal"]:
    assert shots[col].notna().all(), f"Null values in {col} before sort"

shots = shots.sort_values(SORT_COLS, ignore_index=True)

# Deterministic chronological order is defined by SORT_COLS, not just match_datetime_utc.
# Primary key is non-decreasing; subsequent keys break ties deterministically.
assert shots["match_datetime_utc"].is_monotonic_increasing, \
    "Primary sort key not non-decreasing — sort failed"

print(f"Combined attempts: {len(shots):,}")
print(f"Earliest: {shots['match_datetime_utc'].min()}")
print(f"Latest:   {shots['match_datetime_utc'].max()}")

## Step 3: Compute cumulative career counts (vectorised)

- `cumcount()` gives 0, 1, 2, … for each player's rows in global chronological order — the count of prior rows.
- Grouped `cumsum` minus current `is_goal` gives goals scored in all prior rows only.

In [ ]:
shots["career_shots_before_shot"] = (
    shots.groupby("playerId").cumcount().astype("int32")
)

shots["career_goals_before_shot"] = (
    shots.groupby("playerId")["is_goal"]
    .cumsum()
    .sub(shots["is_goal"])
    .astype("int32")
)

shots["career_non_goal_shots_before_shot"] = (
    shots["career_shots_before_shot"] - shots["career_goals_before_shot"]
).astype("int32")

# Dtype contract: int32
assert shots["career_shots_before_shot"].dtype == "int32"
assert shots["career_goals_before_shot"].dtype == "int32"
assert shots["career_non_goal_shots_before_shot"].dtype == "int32"

assert shots["career_shots_before_shot"].ge(0).all(), "Negative career_shots_before_shot"
assert shots["career_goals_before_shot"].ge(0).all(), "Negative career_goals_before_shot"
assert shots["career_non_goal_shots_before_shot"].ge(0).all(), \
    "Negative career_non_goal_shots_before_shot"
assert (shots["career_goals_before_shot"] <= shots["career_shots_before_shot"]).all(), \
    "Goals exceed shots in prior history"

print("Career count columns created and validated")

## Step 4: Compute conversion-rate features

`career_conversion_rate_before_shot` defaults to 0.0 when no prior history exists.
`has_prior_shot_history` disambiguates true zero conversion from no history.

In [ ]:
shots["has_prior_shot_history"] = (
    shots["career_shots_before_shot"] > 0
).astype("int8")

shots["career_conversion_rate_before_shot"] = np.where(
    shots["career_shots_before_shot"] > 0,
    shots["career_goals_before_shot"] / shots["career_shots_before_shot"],
    0.0,
).astype("float32")

# Dtype contract
assert shots["has_prior_shot_history"].dtype == "int8"
assert shots["career_conversion_rate_before_shot"].dtype == "float32"

assert shots["has_prior_shot_history"].isin([0, 1]).all(), \
    "has_prior_shot_history not binary"
assert shots["career_conversion_rate_before_shot"].between(0, 1).all(), \
    "career_conversion_rate_before_shot out of [0, 1]"
assert (shots.loc[
    shots["career_shots_before_shot"] == 0,
    "career_conversion_rate_before_shot"
] == 0.0).all(), "Zero-history rows must have conversion rate = 0.0"

print("Conversion rate and history flag created and validated")

## Step 5: Verify chronology-safe cumulative features

In [ ]:
# First shot for every player must start at zero prior shots and zero prior goals
first_rows = shots.groupby("playerId", sort=False).head(1)
assert (first_rows["career_shots_before_shot"] == 0).all(), \
    "First player shot does not start at 0 prior shots"
assert (first_rows["career_goals_before_shot"] == 0).all(), \
    "First player shot does not start at 0 prior goals"

# career_shots_before_shot must increment by exactly 1 between consecutive player rows
shot_diffs_ok = (
    shots.groupby("playerId")["career_shots_before_shot"]
    .diff()
    .fillna(0)
    .isin([0, 1])
    .all()
)
assert shot_diffs_ok, "career_shots_before_shot does not increment correctly within player"

# career_goals_before_shot must never decrease within a player timeline
goal_non_decreasing = (
    shots.groupby("playerId")["career_goals_before_shot"]
    .diff()
    .fillna(0)
    .ge(0)
    .all()
)
assert goal_non_decreasing, "career_goals_before_shot decreases within player"

# No duplicate shot identities in the combined table
ID_COLS = ["league", "matchId", "playerId", "teamId", "eventSec", "matchPeriod"]
assert not shots[ID_COLS].duplicated().any(), "Duplicate shot identities in combined table"

print("All chronology-safety checks passed")

## Step 6: Summarise player-history coverage

In [ ]:
coverage = shots.groupby("dataset_split").agg(
    n_shots=("is_goal", "count"),
    share_with_prior_history=("has_prior_shot_history", "mean"),
    mean_career_shots_before=("career_shots_before_shot", "mean"),
    mean_career_goals_before=("career_goals_before_shot", "mean"),
    mean_conversion_before=("career_conversion_rate_before_shot", "mean"),
).round(4)
print("Coverage by split:")
display(coverage)

top_players = (
    shots.groupby("playerId")
    .agg(
        n_shots=("is_goal", "count"),
        goals=("is_goal", "sum"),
        max_prior_shots=("career_shots_before_shot", "max"),
    )
    .sort_values("n_shots", ascending=False)
    .head(20)
)
print("\nTop 20 players by attempt count:")
display(top_players)

## Step 7: Split back into train and test

In [ ]:
train_skill = shots[shots["dataset_split"] == "train"].copy()
test_skill  = shots[shots["dataset_split"] == "test"].copy()

assert len(train_skill) == len(train), "Train row count changed after skill computation"
assert len(test_skill)  == len(test),  "Test row count changed after skill computation"
assert set(train_skill["league"].unique()) == {"France", "Germany", "Italy", "Spain"}, \
    "Wrong leagues in train_skill"
assert set(test_skill["league"].unique()) == {"England"}, "Wrong leagues in test_skill"

print(f"Train: {len(train_skill):,}  |  Test: {len(test_skill):,}")

## Step 8: Save skill-enriched datasets

In [ ]:
NEW_SKILL_COLS = [
    "career_shots_before_shot",
    "career_goals_before_shot",
    "career_non_goal_shots_before_shot",
    "career_conversion_rate_before_shot",
    "has_prior_shot_history",
]

# Drop dataset_split — not a feature, not needed downstream.
# All other columns, including xg_pred, are preserved in the saved skill datasets.
SAVE_COLS = [c for c in train_skill.columns if c != "dataset_split"]

train_skill[SAVE_COLS].to_parquet(DATA_DIR / "wyscout_train_xg_skill.parquet", index=False)
test_skill[SAVE_COLS].to_parquet(DATA_DIR  / "wyscout_test_xg_skill.parquet",  index=False)
shots[
    [c for c in SAVE_COLS if c in shots.columns]
].head(100).to_csv(DATA_DIR / "wyscout_xg_skill_sample.csv", index=False)

print(f"Saved {len(train_skill):,} train → wyscout_train_xg_skill.parquet")
print(f"Saved {len(test_skill):,} test  → wyscout_test_xg_skill.parquet")
print("Saved 100-row sample         → wyscout_xg_skill_sample.csv")
print(f"\nNew skill columns ({len(NEW_SKILL_COLS)}): {NEW_SKILL_COLS}")
print(f"Total saved columns: {len(SAVE_COLS)}")

## Step 9: Reload verification

In [ ]:
train_check = pd.read_parquet(DATA_DIR / "wyscout_train_xg_skill.parquet")
test_check  = pd.read_parquet(DATA_DIR / "wyscout_test_xg_skill.parquet")

# xg_pred preserved (contract: passed through from notebook 04)
assert "xg_pred" in train_check.columns, "xg_pred missing from train skill parquet"
assert "xg_pred" in test_check.columns,  "xg_pred missing from test skill parquet"

# dataset_split must not be persisted
assert "dataset_split" not in train_check.columns, "dataset_split should not be saved"
assert "dataset_split" not in test_check.columns,  "dataset_split should not be saved"

# All new skill columns present
required_skill_cols = set(NEW_SKILL_COLS)
assert required_skill_cols.issubset(train_check.columns), \
    f"Missing train skill cols: {required_skill_cols - set(train_check.columns)}"
assert required_skill_cols.issubset(test_check.columns), \
    f"Missing test skill cols: {required_skill_cols - set(test_check.columns)}"

# Row counts
assert len(train_check) == len(train_skill), "Train reload row count mismatch"
assert len(test_check)  == len(test_skill),  "Test reload row count mismatch"

# Dtype checks on both reloaded files (parquet round-trip can widen types)
for df_name, df_check in [("train", train_check), ("test", test_check)]:
    for col in ["career_shots_before_shot", "career_goals_before_shot",
                "career_non_goal_shots_before_shot"]:
        assert pd.api.types.is_integer_dtype(df_check[col]), \
            f"{col} not integer in {df_name}"
    assert pd.api.types.is_integer_dtype(df_check["has_prior_shot_history"]), \
        f"has_prior_shot_history not integer in {df_name}"
    assert pd.api.types.is_float_dtype(df_check["career_conversion_rate_before_shot"]), \
        f"career_conversion_rate_before_shot not float in {df_name}"

# Range checks
assert train_check["career_shots_before_shot"].ge(0).all()
assert test_check["career_shots_before_shot"].ge(0).all()
assert train_check["career_goals_before_shot"].ge(0).all()
assert test_check["career_goals_before_shot"].ge(0).all()
assert train_check["career_conversion_rate_before_shot"].between(0, 1).all()
assert test_check["career_conversion_rate_before_shot"].between(0, 1).all()
assert train_check["has_prior_shot_history"].isin([0, 1]).all()
assert test_check["has_prior_shot_history"].isin([0, 1]).all()

# Aggregate identity checks (stronger than row count alone)
assert train_check["career_shots_before_shot"].sum() == \
    train_skill["career_shots_before_shot"].sum(), \
    "Train career_shots_before_shot sum mismatch on reload"
assert test_check["career_shots_before_shot"].sum() == \
    test_skill["career_shots_before_shot"].sum(), \
    "Test career_shots_before_shot sum mismatch on reload"

print("Reload verification passed")
print(f"Train: {train_check.shape}  |  Test: {test_check.shape}")
display(train_check[[
    "playerId", "is_goal",
    "career_shots_before_shot",
    "career_goals_before_shot",
    "career_conversion_rate_before_shot",
    "has_prior_shot_history",
]].head())